# Source Dimension Loader

Maintains the `warehouse.dim_source` dimension table with incremental refresh capability.

## Purpose
Track data source systems and their metadata for lineage and reporting.

## Key Features
* Stable surrogate keys for source systems (auto-increment)
* Source metadata (URLs, ingestion methods)
* SCD Type 1 (overwrite on change)
* Idempotent: safe to re-run

## Architecture
**Source**: `workspace.silver.silver_jobs_current`  
**Target**: `workspace.warehouse.dim_source`  
**Metadata**: `workspace.metadata.dim_source_refresh_log`  
**Mode**: Incremental (merge new sources, update existing)

## Batch Processing
* Tracks refresh history in metadata table
* Auto-assigns surrogate keys to new sources
* Updates existing sources with latest metadata
* Validates data quality after each refresh

In [0]:
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")
dbutils.widgets.text("default_ingestion_method", "API", "Default Ingestion Method")

FORCE_FULL_REFRESH = dbutils.widgets.get("force_full_refresh") == "true"
DEFAULT_INGESTION_METHOD = dbutils.widgets.get("default_ingestion_method")

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType
import json

CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"
SILVER_SCHEMA = f"{CATALOG}.silver"

SOURCE_TABLE = f"{SILVER_SCHEMA}.silver_jobs_current"
TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_source"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_source_refresh_log"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()

# Source metadata mappings
SOURCE_METADATA = {
    'linkedin': {'url': 'https://www.linkedin.com/jobs', 'method': 'API'},
    'indeed': {'url': 'https://www.indeed.com', 'method': 'API'},
    'glassdoor': {'url': 'https://www.glassdoor.com', 'method': 'SCRAPE'},
    'remotive': {'url': 'https://remotive.com', 'method': 'API'},
    'arbeitnow': {'url': 'https://www.arbeitnow.com', 'method': 'API'},
}

print(f"Run ID: {run_id}")
print(f"Force full refresh: {FORCE_FULL_REFRESH}")
print(f"Default ingestion method: {DEFAULT_INGESTION_METHOD}")

In [0]:
%sql
-- Create source dimension table if not exists
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_source (
  source_sk INT NOT NULL COMMENT 'Surrogate key for source',
  source_name STRING NOT NULL COMMENT 'Source system name',
  source_display_name STRING NOT NULL COMMENT 'Display name',
  source_url STRING COMMENT 'Source website URL',
  ingestion_method STRING NOT NULL COMMENT 'API, SCRAPE, FILE, etc.',
  is_active BOOLEAN NOT NULL COMMENT 'Is source currently active',
  first_seen TIMESTAMP NOT NULL COMMENT 'First time source was seen',
  last_seen TIMESTAMP NOT NULL COMMENT 'Last time source was seen',
  updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
  CONSTRAINT pk_dim_source PRIMARY KEY (source_sk)
)
USING DELTA
COMMENT 'Source system dimension for tracking data lineage';

-- Create metadata tracking table
CREATE TABLE IF NOT EXISTS workspace.metadata.dim_source_refresh_log (
  run_id STRING,
  sources_extracted INT,
  sources_inserted INT,
  sources_updated INT,
  force_full_refresh BOOLEAN,
  processed_at TIMESTAMP,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks source dimension refresh history';

In [0]:
print("Extracting distinct sources from silver layer...", end=" ")

# Extract distinct sources with aggregated metadata
sources_df = spark.sql(f"""
    SELECT DISTINCT
        source_name,
        COUNT(*) OVER (PARTITION BY source_name) as job_count,
        MIN(created_at) OVER (PARTITION BY source_name) as first_seen,
        MAX(updated_at) OVER (PARTITION BY source_name) as last_seen
    FROM {SOURCE_TABLE}
    WHERE source_name IS NOT NULL
    AND is_active = true
    AND soft_delete_flag = false
""")

sources_count = sources_df.count()
print(f"✓ Found {sources_count} distinct sources")

# Get current max surrogate key
max_sk_result = spark.sql(f"SELECT COALESCE(MAX(source_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
max_sk = max_sk_result[0]['max_sk']

print(f"Current max surrogate key: {max_sk}")

In [0]:
# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("sources_extracted", IntegerType(), True),
    StructField("sources_inserted", IntegerType(), True),
    StructField("sources_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

try:
    print(f"Processing sources into {TARGET_TABLE}...", end=" ")
    
    # Get existing sources for comparison
    existing_sources = spark.sql(f"SELECT source_name, source_sk FROM {TARGET_TABLE}")
    
    # Join to assign keys (existing or new)
    sources_with_keys = sources_df.alias("s").join(
        existing_sources.alias("e"),
        F.col("s.source_name") == F.col("e.source_name"),
        "left"
    )
    
    # Assign surrogate keys
    from pyspark.sql.window import Window
    window_spec = Window.orderBy("s.source_name")
    
    sources_final = sources_with_keys.withColumn(
        "source_sk",
        F.coalesce(F.col("e.source_sk"), F.lit(max_sk) + F.row_number().over(window_spec))
    ).withColumn(
        "source_display_name",
        F.initcap(F.col("s.source_name"))
    ).withColumn(
        "source_url",
        F.when(F.col("s.source_name") == "linkedin", F.lit(SOURCE_METADATA.get('linkedin', {}).get('url')))
         .when(F.col("s.source_name") == "indeed", F.lit(SOURCE_METADATA.get('indeed', {}).get('url')))
         .when(F.col("s.source_name") == "glassdoor", F.lit(SOURCE_METADATA.get('glassdoor', {}).get('url')))
         .when(F.col("s.source_name") == "remotive", F.lit(SOURCE_METADATA.get('remotive', {}).get('url')))
         .when(F.col("s.source_name") == "arbeitnow", F.lit(SOURCE_METADATA.get('arbeitnow', {}).get('url')))
         .otherwise(None)
    ).withColumn(
        "ingestion_method",
        F.when(F.col("s.source_name") == "linkedin", F.lit(SOURCE_METADATA.get('linkedin', {}).get('method')))
         .when(F.col("s.source_name") == "indeed", F.lit(SOURCE_METADATA.get('indeed', {}).get('method')))
         .when(F.col("s.source_name") == "glassdoor", F.lit(SOURCE_METADATA.get('glassdoor', {}).get('method')))
         .when(F.col("s.source_name") == "remotive", F.lit(SOURCE_METADATA.get('remotive', {}).get('method')))
         .when(F.col("s.source_name") == "arbeitnow", F.lit(SOURCE_METADATA.get('arbeitnow', {}).get('method')))
         .otherwise(F.lit(DEFAULT_INGESTION_METHOD))
    ).withColumn(
        "is_active", F.lit(True)
    ).withColumn(
        "updated_at", F.lit(run_timestamp)
    ).select(
        "source_sk",
        F.col("s.source_name").alias("source_name"),
        "source_display_name",
        "source_url",
        "ingestion_method",
        "is_active",
        F.col("s.first_seen").alias("first_seen"),
        F.col("s.last_seen").alias("last_seen"),
        "updated_at"
    )
    
    # Create temp view for merge
    sources_final.createOrReplaceTempView("sources_to_merge")
    
    # Check if table schema matches expected schema
    existing_cols = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]
    expected_cols = ['source_sk', 'source_name', 'source_display_name', 'source_url', 'ingestion_method', 'is_active', 'first_seen', 'last_seen', 'updated_at']
    schema_matches = set(existing_cols) == set(expected_cols)
    
    if FORCE_FULL_REFRESH or not schema_matches:
        # Full refresh: drop and recreate with new schema
        if not schema_matches:
            print(f"Schema mismatch detected. Performing full refresh...")
            spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
            spark.sql(f"""
            CREATE TABLE {TARGET_TABLE} (
              source_sk INT NOT NULL COMMENT 'Surrogate key for source',
              source_name STRING NOT NULL COMMENT 'Source system name',
              source_display_name STRING NOT NULL COMMENT 'Display name',
              source_url STRING COMMENT 'Source website URL',
              ingestion_method STRING NOT NULL COMMENT 'API, SCRAPE, FILE, etc.',
              is_active BOOLEAN NOT NULL COMMENT 'Is source currently active',
              first_seen TIMESTAMP NOT NULL COMMENT 'First time source was seen',
              last_seen TIMESTAMP NOT NULL COMMENT 'Last time source was seen',
              updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
              CONSTRAINT pk_dim_source PRIMARY KEY (source_sk)
            )
            USING DELTA
            COMMENT 'Source system dimension for tracking data lineage'
            """)
        else:
            spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        
        sources_final.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        sources_inserted = sources_final.count()
        sources_updated = 0
        print(f"✓ Full refresh: {sources_inserted} sources inserted")
    else:
        # Incremental: merge
        merge_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING sources_to_merge source
        ON target.source_name = source.source_name
        WHEN MATCHED THEN UPDATE SET
            target.source_display_name = source.source_display_name,
            target.source_url = source.source_url,
            target.ingestion_method = source.ingestion_method,
            target.is_active = source.is_active,
            target.last_seen = source.last_seen,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
        """
        
        spark.sql(merge_sql)
        
        # Count metrics
        sources_inserted = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM sources_to_merge
            WHERE source_name NOT IN (SELECT source_name FROM {TARGET_TABLE})
        """).collect()[0]['cnt']
        
        sources_updated = sources_count - sources_inserted
        
        print(f"✓ Merge complete: {sources_inserted} new, {sources_updated} updated")
    
    # Log to metadata
    metadata_data = [(
        run_id,
        sources_count,
        sources_inserted,
        sources_updated,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'success',
        None
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    result = {
        "status": "success",
        "run_id": run_id,
        "sources_extracted": sources_count,
        "sources_inserted": sources_inserted,
        "sources_updated": sources_updated,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(result, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"✗ Error: {error_msg}")
    
    # Log failure to metadata
    metadata_data = [(
        run_id,
        sources_count if 'sources_count' in locals() else 0,
        0,
        0,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'failed',
        error_msg
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate source dimension
SELECT 
  source_sk,
  source_name,
  source_display_name,
  source_url,
  ingestion_method,
  is_active,
  first_seen,
  last_seen,
  updated_at
FROM workspace.warehouse.dim_source
ORDER BY source_sk;

-- Show refresh history
SELECT 
  run_id,
  sources_extracted,
  sources_inserted,
  sources_updated,
  force_full_refresh,
  processed_at,
  status
FROM workspace.metadata.dim_source_refresh_log
ORDER BY processed_at DESC
LIMIT 10;